# Régression linéaire : de la formule à scikit-learn

**Objectifs.** Ajuster une droite par gradient descent, comparer à `LinearRegression`, utiliser un dataset de régression intégré, et interpréter un résidu. Durée : 90 min.

## 1. Intuition et modèle

La régression prédit une quantité numérique. Pour un seul attribut, `y_hat = w*x+b` ; `w` est la pente et `b` l’intercept. MSE choisit les coefficients en pénalisant les grandes erreurs au carré.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(8)
x = np.linspace(0, 10, 100)
y = 4.5 * x + 7 + rng.normal(0, 3, x.size)
plt.scatter(x, y, s=18); plt.xlabel('Heures de pratique'); plt.ylabel('Score simulé')
plt.title('Données synthétiques'); plt.show()

### Quel rôle jouent la pente et l'intercept ?

Sur deux points de démonstration `(0,1)` et `(1,3)`, la droite `w=2,b=1` prédit exactement les deux valeurs : MSE=0. La pente contrôle l'augmentation de `ŷ` pour une unité supplémentaire de `x`, et le biais décale toute la droite. Le graphique montre qu'une même pente avec un autre biais ne décrit plus les mêmes points.

In [ ]:
small_x = np.array([0., 1.]); small_y = np.array([1., 3.])
candidate_x = np.linspace(-0.2, 1.2, 50)
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.scatter(small_x, small_y, s=70, color='black', zorder=3, label='Deux observations')
for slope, intercept, label in [(2, 1, 'w=2, b=1'), (2, 0, 'w=2, b=0'), (1, 1, 'w=1, b=1')]:
    ax.plot(candidate_x, slope * candidate_x + intercept, label=label)
ax.set(xlabel='x', ylabel='ŷ', title='Effet de la pente et du biais')
ax.legend(); ax.grid(alpha=0.2); plt.show()
print('MSE exacte :', np.mean((2 * small_x + 1 - small_y) ** 2))

## 2. Calculer une régression à la main

La dérivée de MSE indique comment modifier chacun des paramètres. On part de `w=0,b=0`, puis on répète la mise à jour. L’implémentation ci-dessous n’utilise aucune librairie ML.

In [ ]:
weight, bias = 0.0, 0.0
learning_rate = 0.01
losses = []
for _ in range(800):
    prediction = weight * x + bias
    error = prediction - y
    losses.append(np.mean(error ** 2))
    grad_w = 2 * np.mean(error * x)
    grad_b = 2 * np.mean(error)
    weight -= learning_rate * grad_w
    bias -= learning_rate * grad_b
print(f'pente={weight:.2f}; intercept={bias:.2f}; MSE={losses[-1]:.2f}')

## 3. Visualiser et inspecter les résidus

Un résidu est `observed - predicted`. Une structure courbe dans les résidus peut indiquer que la droite manque une relation non linéaire ; une forme en éventail peut indiquer une variance non constante.

In [ ]:
predictions = weight * x + bias
residuals = y - predictions
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(x, y, s=16); axes[0].plot(x, predictions, color='darkorange')
axes[0].set(xlabel='x', ylabel='y', title='Droite ajustée')
axes[1].scatter(predictions, residuals, s=16); axes[1].axhline(0, color='black', linewidth=1)
axes[1].set(xlabel='Prédiction', ylabel='Résidu', title='Diagnostic des erreurs')
plt.tight_layout(); plt.show()

## 4. Comparer à l’implémentation de bibliothèque

`scikit-learn` facilite la composition mais n’enlève pas la nécessité de choisir un split, des variables et des métriques. Nous reproduisons la comparaison sur les mêmes points.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(x.reshape(-1, 1), y)
print('pente/intercept scikit-learn :', model.coef_[0].round(3), model.intercept_.round(3))
print('différence avec la descente :', round(weight - model.coef_[0], 4), round(bias - model.intercept_, 4))

## 5. Tester sur un problème tabulaire intégré

Le dataset diabetes est distribué avec scikit-learn. Un split reproductible donne un score de démonstration ; le nombre dépend du split. L’évaluation plus soigneuse par cross-validation arrive au module 05.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

features, target = load_diabetes(return_X_y=True, as_frame=True)
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.25, random_state=42)
regressor = LinearRegression().fit(x_train, y_train)
prediction = regressor.predict(x_test)
print('MAE :', round(mean_absolute_error(y_test, prediction), 2))
print('RMSE :', round(mean_squared_error(y_test, prediction) ** 0.5, 2))
print('R² :', round(r2_score(y_test, prediction), 3))

### Un score n'a de sens qu'avec un point de départ

`DummyRegressor` apprend seulement la moyenne de la cible train. Sa MAE fournit le coût d'un modèle trivial sous les mêmes conditions de split. MAE est en unités de la cible ; R² compare à une prédiction constante sur le test et peut être négatif. Le graphique compare les MAE sans masquer l'échelle des erreurs.

In [ ]:
from sklearn.dummy import DummyRegressor
baseline = DummyRegressor(strategy='mean').fit(x_train, y_train)
baseline_mae = mean_absolute_error(y_test, baseline.predict(x_test))
model_mae = mean_absolute_error(y_test, prediction)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.bar(['Moyenne train', 'Régression linéaire'], [baseline_mae, model_mae], color=['tab:gray', 'tab:blue'])
ax.set(ylabel='MAE (unités de la cible)', title='Comparaison sur le même test')
for index, value in enumerate([baseline_mae, model_mae]):
    ax.text(index, value, f'{value:.1f}', ha='center', va='bottom')
plt.show()
print('Amélioration de MAE :', round(baseline_mae - model_mae, 2))

**Limite du protocole :** un seul split ne quantifie pas la variation du score entre échantillons. Pour comparer plusieurs choix de modèle, utiliser la validation croisée sur le train, garder le test pour le résultat final, puis inspecter les résidus par sous-groupe pertinent.

## Limites, conclusion et exercice

Une relation linéaire peut mal représenter des effets non linéaires, interactions ou mesures biaisées. Un coefficient n’est causal que sous des hypothèses bien plus fortes. Comparez au baseline `DummyRegressor`, et testez une transformation polynomial sans regarder le test pour choisir son degré.

Exercices et correction séparés : [`04_models.ipynb`](../../exercises/intermediate/04_models.ipynb).